# MedTrack_DV - Hospital Data Cleaning & Transformation Notebook
**Module 2 Deliverable: Data Cleaning & Transformation**

This notebook loads the raw hospital operations dataset (`hospital_raw_data.csv`), cleans duplicate records, handles missing data, standardizes department names, normalizes healthcare metrics, and exports a Tableau-ready cleaned dataset (`hospital_cleaned.csv`).

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

# Locate data paths
raw_path = 'data/hospital_raw_data.csv' if os.path.exists('data/hospital_raw_data.csv') else 'hospital_raw_data.csv'
print(f"Loading raw data from: {raw_path}")
df_raw = pd.read_csv(raw_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 1. Inspect Raw Data Quality & Missing Values

In [2]:
print("=== Missing Values Summary in Raw Data ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
print(missing_df[missing_df['Missing_Count'] > 0])

initial_missing_pct = (df_raw.isnull().sum().sum() / df_raw.size) * 100
print(f"\nOverall Raw Dataset Missing Value Rate: {initial_missing_pct:.2f}%")

## 2. Deduplicate Records

In [3]:
initial_rows = len(df_raw)
df_clean = df_raw.drop_duplicates().copy()
duplicates_removed = initial_rows - len(df_clean)
print(f"Removed {duplicates_removed} duplicate records. Clean records count: {len(df_clean)}")

## 3. Standardize Department Names & Text Formatting

In [4]:
print("Departments before standardization:", df_clean['Department'].unique())

# Standardize mapping
dept_mapping = {
    'cardiology': 'Cardiology',
    'ICU ': 'ICU',
    'Pediatrics ': 'Pediatrics',
    'general medicine': 'General Medicine',
    'SURGERY': 'Surgery'
}

df_clean['Department'] = df_clean['Department'].astype(str).str.strip()
df_clean['Department'] = df_clean['Department'].replace(dept_mapping)
print("Departments after standardization:", df_clean['Department'].unique())

## 4. Handle Missing Values

In [5]:
# Convert dates to datetime
df_clean['Admission_Date'] = pd.to_datetime(df_clean['Admission_Date'])
df_clean['Discharge_Date'] = pd.to_datetime(df_clean['Discharge_Date'])

# Impute missing Discharge_Date based on Admission_Date + Length_of_Stay
missing_dis = df_clean['Discharge_Date'].isnull()
df_clean.loc[missing_dis, 'Discharge_Date'] = df_clean.loc[missing_dis, 'Admission_Date'] + pd.to_timedelta(df_clean.loc[missing_dis, 'Length_of_Stay'], unit='D')

# Impute Equipment_Used with 'None' if missing
df_clean['Equipment_Used'] = df_clean['Equipment_Used'].fillna('None')

# Impute Satisfaction_Score with Department median
dept_median_sat = df_clean.groupby('Department')['Satisfaction_Score'].transform('median')
df_clean['Satisfaction_Score'] = df_clean['Satisfaction_Score'].fillna(dept_median_sat).round(1)

# Impute Readmitted_30_Days based on Severity logic or mode
df_clean['Readmitted_30_Days'] = df_clean['Readmitted_30_Days'].fillna('No')

# Verify missing values after imputation
final_missing_pct = (df_clean.isnull().sum().sum() / df_clean.size) * 100
print(f"Missing Value Rate after Cleaning: {final_missing_pct:.2f}% (Target < 2%)")
assert final_missing_pct < 2.0, "Missing value evaluation failed!"

## 5. Normalize Healthcare Operational Indicators & Derived Features

In [6]:
# Format dates as YYYY-MM-DD string for Tableau
df_clean['Admission_Date_Str'] = df_clean['Admission_Date'].dt.strftime('%Y-%m-%d')
df_clean['Discharge_Date_Str'] = df_clean['Discharge_Date'].dt.strftime('%Y-%m-%d')
df_clean['Admission_Month'] = df_clean['Admission_Date'].dt.strftime('%Y-%m')

# Ensure Length_of_Stay is correct
df_clean['Length_of_Stay'] = (df_clean['Discharge_Date'] - df_clean['Admission_Date']).dt.days
df_clean['Length_of_Stay'] = df_clean['Length_of_Stay'].apply(lambda x: max(x, 1))

# Daily Treatment Cost
df_clean['Daily_Treatment_Cost'] = (df_clean['Treatment_Cost'] / df_clean['Length_of_Stay']).round(2)

# Department Efficiency Score (synthetic normalized metric based on bed utilization and satisfaction)
df_clean['Dept_Efficiency_Score'] = ((df_clean['Bed_Utilization_Pct'] * 0.6) + (df_clean['Satisfaction_Score'] * 20 * 0.4)).round(2)

print("=== Operational Summary Statistics ===")
print(df_clean[['Length_of_Stay', 'Treatment_Cost', 'Daily_Treatment_Cost', 'Bed_Utilization_Pct', 'Dept_Efficiency_Score']].describe())

## 6. Export Cleaned Dataset to CSV

In [7]:
# Prepare export dataframe
df_export = df_clean.copy()
df_export['Admission_Date'] = df_export['Admission_Date_Str']
df_export['Discharge_Date'] = df_export['Discharge_Date_Str']
df_export = df_export.drop(columns=['Admission_Date_Str', 'Discharge_Date_Str'])

# Define target output paths
os.makedirs('data', exist_ok=True)
out_data_path = 'data/hospital_cleaned.csv'
out_root_path = 'hospital_cleaned.csv'

df_export.to_csv(out_data_path, index=False)
df_export.to_csv(out_root_path, index=False)

print(f"Successfully saved cleaned dataset to '{out_data_path}' and '{out_root_path}'")
print(f"Total records: {len(df_export)}, Columns: {len(df_export.columns)}")